In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

project_root = Path.cwd().parent
sys.path.append(str(project_root))

print(project_root)

c:\Users\Satvika Putcha\OneDrive\Dokumen\equity-risk-assessment-framework


In [2]:
daily_returns = pd.read_csv(
    project_root / "data" / "processed" / "daily_returns.csv",
    index_col=0,
    parse_dates=True
)

portfolio_assets = ["AAPL", "JPM", "JNJ", "EFA", "LQD", "DBC"]
benchmark = "SPY"

expected_columns = portfolio_assets + [benchmark]

assert daily_returns.columns.tolist() == expected_columns, (
    f"Unexpected columns: {daily_returns.columns.tolist()}"
)

print("Dataset shape:", daily_returns.shape)
display(daily_returns.head())

Dataset shape: (2479, 7)


,AAPL,JPM,JNJ,EFA,LQD,DBC,SPY
Date,,,,,,,
2017-01-04,-0.001119,0.001845,-0.001640,0.011362,0.003411,0.007668,0.005949
2017-01-05,0.005085,-0.009205,0.010463,0.008510,0.005353,0.005073,-0.000795
2017-01-06,0.011148,0.000116,-0.004792,-0.003375,-0.005240,-0.001262,0.003578
2017-01-09,0.009160,0.000697,-0.000172,-0.001524,0.001954,-0.015161,-0.003301
2017-01-10,0.001009,0.002901,-0.001032,0.000170,-0.000594,-0.004490,0.000000


Annual Returns

In [3]:
annual_returns = (1+daily_returns).prod() ** (252 / len(daily_returns)) -1

annual_returns

AAPL    0.300924
JPM     0.180113
JNJ     0.125164
EFA     0.092117
LQD     0.022968
DBC     0.090084
SPY     0.148765
dtype: float64

Annual Volatility

In [4]:
annual_volatility = daily_returns.std() * np.sqrt(252)

annual_volatility

AAPL    0.293353
JPM     0.274770
JNJ     0.191177
EFA     0.171400
LQD     0.087430
DBC     0.180228
SPY     0.181329
dtype: float64

Sharpe Ratio

In [5]:
fred_url = (
    "https://fred.stlouisfed.org/graph/fredgraph.csv"
    "?id=DGS3MO&cosd=2017-01-01"
)

risk_free_data = pd.read_csv(
    fred_url,
    index_col=0,
    parse_dates=True
)

risk_free_data["DGS3MO"] = pd.to_numeric(
    risk_free_data["DGS3MO"],
    errors="coerce"
)

risk_free_rate = risk_free_data["DGS3MO"].reindex(
    daily_returns.index
).ffill()

assert risk_free_rate.notna().all()
assert risk_free_rate.index.equals(daily_returns.index)

display(risk_free_rate.head())
print("Latest annual risk-free yield:", risk_free_rate.iloc[-1], "%")

Date
2017-01-04    0.53
2017-01-05    0.52
2017-01-06    0.53
2017-01-09    0.50
2017-01-10    0.52
Name: DGS3MO, dtype: float64

Latest annual risk-free yield: 4.32 %


In [7]:
daily_risk_free_rate = (risk_free_rate / 100) / 252

daily_excess_returns = daily_returns.sub(
    daily_risk_free_rate,
    axis=0
)

sharpe_ratio = (
    daily_excess_returns.mean()
    / daily_excess_returns.std()
) * np.sqrt(252)

display(sharpe_ratio.sort_values(ascending=False).round(3))

AAPL    0.955
SPY     0.713
JPM     0.646
JNJ     0.578
EFA     0.450
DBC     0.426
LQD     0.008
dtype: float64

Correlation Matrix

In [8]:
correlation_matrix = daily_returns.corr()
correlation_matrix

,AAPL,JPM,JNJ,EFA,LQD,DBC,SPY
AAPL,1.000000,0.409687,0.292897,0.579429,0.239809,0.184361,0.739529
JPM,0.409687,1.000000,0.329787,0.650571,0.132056,0.227348,0.696140
JNJ,0.292897,0.329787,1.000000,0.368315,0.128090,0.063551,0.410255
EFA,0.579429,0.650571,0.368315,1.000000,0.350858,0.290326,0.847062
LQD,0.239809,0.132056,0.128090,0.350858,1.000000,0.044128,0.316102
DBC,0.184361,0.227348,0.063551,0.290326,0.044128,1.000000,0.287465
SPY,0.739529,0.696140,0.410255,0.847062,0.316102,0.287465,1.000000


Portfolio Returns

In [9]:
weights = pd.Series(
    1 / len(portfolio_assets),
    index=portfolio_assets,
    name="Portfolio Weight"
)

# Validate portfolio weights
assert len(weights) == 6
assert np.isclose(weights.sum(), 1.0)
assert weights.index.tolist() == portfolio_assets

# Select only the six portfolio holdings
asset_returns = daily_returns[portfolio_assets]

# Calculate daily constant-weight portfolio returns
portfolio_returns = asset_returns.dot(weights)

print("Portfolio weights:")
display(weights)

print("\nTotal portfolio weight:", weights.sum())

print("\nPortfolio returns:")
display(portfolio_returns.head())

Portfolio weights:


AAPL    0.166667
JPM     0.166667
JNJ     0.166667
EFA     0.166667
LQD     0.166667
DBC     0.166667
Name: Portfolio Weight, dtype: float64


Total portfolio weight: 0.9999999999999999

Portfolio returns:


Date
2017-01-04    0.003588
2017-01-05    0.004213
2017-01-06   -0.000568
2017-01-09   -0.000841
2017-01-10   -0.000340
dtype: float64

Portfolio Annual Return

In [10]:
portfolio_annual_return = (1 + portfolio_returns).prod() ** (252 / len(portfolio_returns)) -1

print(f"Portfolio Annual Returns: {portfolio_annual_return:.2%}")


Portfolio Annual Returns: 14.72%


Portfolio Annual Volatility

In [11]:
portfolio_annual_volatility = portfolio_returns.std() * np.sqrt(252)

print(f"Porfolio Annual Volatility: {portfolio_annual_volatility:.2%}")

Porfolio Annual Volatility: 13.37%


Portfolio Sharpe Ratio

In [12]:
portfolio_excess_returns = (
    portfolio_returns - daily_risk_free_rate
)

portfolio_sharpe = (
    portfolio_excess_returns.mean()
    / portfolio_excess_returns.std()
) * np.sqrt(252)

print(f"Portfolio Sharpe Ratio: {portfolio_sharpe:.3f}")

Portfolio Sharpe Ratio: 0.900


Portfolio Value

In [13]:
portfolio_value = (1 + portfolio_returns).cumprod()

portfolio_value.head()

Date
2017-01-04    1.003588
2017-01-05    1.007816
2017-01-06    1.007244
2017-01-09    1.006397
2017-01-10    1.006055
dtype: float64

Running Maximum

In [14]:
running_max = portfolio_value.cummax()

running_max.head()

Date
2017-01-04    1.003588
2017-01-05    1.007816
2017-01-06    1.007816
2017-01-09    1.007816
2017-01-10    1.007816
dtype: float64

Drawdown

In [15]:
drawdown = (portfolio_value - running_max) / running_max

drawdown.head()

Date
2017-01-04    0.000000
2017-01-05    0.000000
2017-01-06   -0.000568
2017-01-09   -0.001408
2017-01-10   -0.001747
dtype: float64

Maximum Drawdown

In [16]:
max_drawdown = drawdown.min()

print(f"Maximum Drawdown: {max_drawdown:.2%}")

Maximum Drawdown: -28.52%


Value at Risk

In [17]:
var_95 = portfolio_returns.quantile(0.05)

print(f"95% Daily VaR: {var_95:.2%}")

95% Daily VaR: -1.19%


Load Market Returns(SPY)

In [18]:
market_returns = daily_returns["SPY"]

market_returns.head()

Date
2017-01-04    0.005949
2017-01-05   -0.000795
2017-01-06    0.003578
2017-01-09   -0.003301
2017-01-10    0.000000
Name: SPY, dtype: float64

In [19]:
type(market_returns)

pandas.core.series.Series

Market Variance

In [20]:
market_variance = market_returns.var()

print("Market Variance: ",market_variance)

Market Variance:  0.0001304765118518038


Beta for each Asset

In [21]:
betas = {}

for asset in portfolio_assets:
    covariance = daily_returns[asset].cov(market_returns)
    beta = covariance / market_variance
    betas[asset] = beta

beta = pd.Series(betas, name="Beta")

display(beta.sort_values(ascending=False).round(3))

AAPL    1.196
JPM     1.055
EFA     0.801
JNJ     0.433
DBC     0.286
LQD     0.152
Name: Beta, dtype: float64

Asset Summary Table

In [22]:
asset_summary = pd.DataFrame({
    "Annual Return": annual_returns,
    "Annual Volatility": annual_volatility,
    "Sharpe Ratio": sharpe_ratio,
    "Beta": beta.reindex(annual_returns.index)
})

display(asset_summary.round(4))

,Annual Return,Annual Volatility,Sharpe Ratio,Beta
AAPL,0.3009,0.2934,0.9555,1.1964
JPM,0.1801,0.2748,0.6460,1.0549
JNJ,0.1252,0.1912,0.5775,0.4325
EFA,0.0921,0.1714,0.4495,0.8007
LQD,0.0230,0.0874,0.0077,0.1524
DBC,0.0901,0.1802,0.4257,0.2857
SPY,0.1488,0.1813,0.7134,NaN


Save Asset Summary

In [28]:
output_path = (
    project_root / "data" / "processed" / "asset_summary.csv"
)

asset_summary.index.name = "Asset"

asset_summary.to_csv(output_path)

print("asset_summary.csv saved!")
print("Saved to:", output_path)
print("File exists:", output_path.exists())

asset_summary.csv saved!
Saved to: c:\Users\Satvika Putcha\OneDrive\Dokumen\equity-risk-assessment-framework\data\processed\asset_summary.csv
File exists: True


Export Correlation Matrix

In [29]:
correlation_matrix.index.name = "Asset"

correlation_output_path = (
    project_root / "data" / "processed" / "correlation_matrix.csv"
)

correlation_matrix.to_csv(correlation_output_path)

print("correlation_matrix.csv saved!")
print("Saved to:", correlation_output_path)
print("File exists:", correlation_output_path.exists())

correlation_matrix.csv saved!
Saved to: c:\Users\Satvika Putcha\OneDrive\Dokumen\equity-risk-assessment-framework\data\processed\correlation_matrix.csv
File exists: True


Create Portfolio History

In [30]:
portfolio_history = pd.DataFrame({
    "Portfolio Value": portfolio_value,
    "Running Max": running_max,
    "Drawdown": drawdown
})

display(portfolio_history.head())

,Portfolio Value,Running Max,Drawdown
Date,,,
2017-01-04,1.003588,1.003588,0.000000
2017-01-05,1.007816,1.007816,0.000000
2017-01-06,1.007244,1.007816,-0.000568
2017-01-09,1.006397,1.007816,-0.001408
2017-01-10,1.006055,1.007816,-0.001747


Export Portfolio History

In [31]:
portfolio_history.index.name = "Date"

history_output_path = (
    project_root / "data" / "processed" / "portfolio_history.csv"
)

portfolio_history.to_csv(history_output_path)

print("portfolio_history.csv saved!")
print("Saved to:", history_output_path)
print("File exists:", history_output_path.exists())

portfolio_history.csv saved!
Saved to: c:\Users\Satvika Putcha\OneDrive\Dokumen\equity-risk-assessment-framework\data\processed\portfolio_history.csv
File exists: True


Create Portfolio SUmmary

In [32]:
portfolio_summary = pd.DataFrame({
    "Annual Return": [portfolio_annual_return],
    "Annual Volatility": [portfolio_annual_volatility],
    "Sharpe Ratio": [portfolio_sharpe],
    "Maximum Drawdown": [max_drawdown],
    "95% VaR": [var_95]
})

display(portfolio_summary.round(4))

,Annual Return,Annual Volatility,Sharpe Ratio,Maximum Drawdown,95% VaR
0,0.1472,0.1337,0.9005,-0.2852,-0.0119


Export Portfolio Summary

In [33]:
portfolio_summary_output_path = (
    project_root / "data" / "processed" / "portfolio_summary.csv"
)

portfolio_summary.to_csv(
    portfolio_summary_output_path,
    index=False
)

print("portfolio_summary.csv saved!")
print("Saved to:", portfolio_summary_output_path)
print("File exists:", portfolio_summary_output_path.exists())

portfolio_summary.csv saved!
Saved to: c:\Users\Satvika Putcha\OneDrive\Dokumen\equity-risk-assessment-framework\data\processed\portfolio_summary.csv
File exists: True


Power bi correlation heatmap

In [34]:
correlation_long = (
    correlation_matrix
    .rename_axis("Asset1")
    .reset_index()
    .melt(
        id_vars="Asset1",
        var_name="Asset2",
        value_name="Correlation"
    )
)

display(correlation_long.head())

,Asset1,Asset2,Correlation
0,AAPL,AAPL,1.000000
1,JPM,AAPL,0.409687
2,JNJ,AAPL,0.292897
3,EFA,AAPL,0.579429
4,LQD,AAPL,0.239809


In [35]:
correlation_long_output_path = (
    project_root / "data" / "processed" / "correlation_long.csv"
)

correlation_long.to_csv(
    correlation_long_output_path,
    index=False
)

print("correlation_long.csv saved!")
print("Saved to:", correlation_long_output_path)
print("File exists:", correlation_long_output_path.exists())
print("Number of rows:", len(correlation_long))

correlation_long.csv saved!
Saved to: c:\Users\Satvika Putcha\OneDrive\Dokumen\equity-risk-assessment-framework\data\processed\correlation_long.csv
File exists: True
Number of rows: 49


In [36]:
output_files = [
    "asset_summary.csv",
    "correlation_matrix.csv",
    "portfolio_history.csv",
    "portfolio_summary.csv",
    "correlation_long.csv"
]

for filename in output_files:
    file_path = project_root / "data" / "processed" / filename
    exported_data = pd.read_csv(file_path)

    print(f"\n{filename}")
    print("Shape:", exported_data.shape)
    print("Columns:", exported_data.columns.tolist())
    print("Missing values:", exported_data.isna().sum().sum())


asset_summary.csv
Shape: (7, 5)
Columns: ['Asset', 'Annual Return', 'Annual Volatility', 'Sharpe Ratio', 'Beta']
Missing values: 1

correlation_matrix.csv
Shape: (7, 8)
Columns: ['Asset', 'AAPL', 'JPM', 'JNJ', 'EFA', 'LQD', 'DBC', 'SPY']
Missing values: 0

portfolio_history.csv
Shape: (2479, 4)
Columns: ['Date', 'Portfolio Value', 'Running Max', 'Drawdown']
Missing values: 0

portfolio_summary.csv
Shape: (1, 5)
Columns: ['Annual Return', 'Annual Volatility', 'Sharpe Ratio', 'Maximum Drawdown', '95% VaR']
Missing values: 0

correlation_long.csv
Shape: (49, 3)
Columns: ['Asset1', 'Asset2', 'Correlation']
Missing values: 0


In [37]:
# Reload exported datasets
asset_check = pd.read_csv(
    project_root / "data" / "processed" / "asset_summary.csv"
)

corr_check = pd.read_csv(
    project_root / "data" / "processed" / "correlation_matrix.csv",
    index_col="Asset"
)

history_check = pd.read_csv(
    project_root / "data" / "processed" / "portfolio_history.csv",
    parse_dates=["Date"]
)

summary_check = pd.read_csv(
    project_root / "data" / "processed" / "portfolio_summary.csv"
)

long_check = pd.read_csv(
    project_root / "data" / "processed" / "correlation_long.csv"
)

# Validate asset summary
assert asset_check["Asset"].tolist() == portfolio_assets + [benchmark]
assert asset_check["Beta"].isna().sum() == 1

# Validate correlation matrix
assert corr_check.shape == (7, 7)
assert np.allclose(corr_check.values, corr_check.values.T)
assert np.allclose(np.diag(corr_check), 1.0)

# Validate portfolio history
assert len(history_check) == len(portfolio_returns)
assert history_check["Date"].is_monotonic_increasing

# Validate portfolio summary
assert np.isclose(
    summary_check.loc[0, "Annual Return"],
    portfolio_annual_return
)
assert np.isclose(
    summary_check.loc[0, "Annual Volatility"],
    portfolio_annual_volatility
)

# Validate long-format correlation data
assert len(long_check) == 49
assert long_check[["Asset1", "Asset2"]].drop_duplicates().shape[0] == 49

print("All data-integrity checks passed!")

All data-integrity checks passed!
